In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score, confusion_matrix
import xgboost as xgb
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier
import shap

# Load the dataset
data = pd.read_csv('/content/sample_data/Spotify Most Streamed Songs.csv')

# Initial Data Overview
print("Initial Data Overview:")
print(data.describe())
print(data.info())

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/Spotify Most Streamed Songs.csv'

In [ ]:
# Preprocessing: Converting strings to floats and cleaning data
def clean_and_convert(column):
    return pd.to_numeric(column.str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

# Numeric columns expected to be in specific format
numeric_columns = ['streams', 'bpm', 'danceability_%', 'energy_%', 'valence_%', 'acousticness_%', 'speechiness_%']
for col in numeric_columns:
    if col in data.columns:
        data[col] = clean_and_convert(data[col].astype(str))

# Impute NaN values with median
for col in numeric_columns:
    if col in data.columns:
        data[col] = data[col].fillna(data[col].median())

In [ ]:
# EDA: Histograms and correlation matrix
data[numeric_columns].hist(bins=15, figsize=(15, 10), layout=(3, 3))
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(data[numeric_columns].corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Encoding categorical variables
categorical_columns = [col for col in data.columns if data[col].dtype == 'object' and col not in ['track_name', 'artist(s)_name', 'cover_url']]
encoder = OneHotEncoder()
encoded_features = encoder.fit_transform(data[categorical_columns])
encoded_feature_names = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_features.toarray(), columns=encoded_feature_names)
data = pd.concat([data.drop(categorical_columns, axis=1), encoded_df], axis=1)

In [ ]:
# Define features and target
columns_to_drop = ['track_name', 'artist(s)_name', 'cover_url']
X = data.drop(columns_to_drop, axis=1)
y = (data['streams'] >= data['streams'].quantile(0.75)).astype(int)

# Addressing class imbalance
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights_dict = dict(enumerate(class_weights))

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

In [ ]:
# Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Model training with hyperparameter tuning for RandomForest and MLP (neural network)
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
rf_model = RandomForestClassifier(random_state=42, class_weight=class_weights_dict)
grid_rf = GridSearchCV(rf_model, param_grid_rf, cv=3, scoring='accuracy')
grid_rf.fit(X_train_scaled, y_train)
print(f"Best parameters for Random Forest: {grid_rf.best_params_}")

param_grid_mlp = {
    'hidden_layer_sizes': [(50,), (100,)],
    'activation': ['tanh', 'relu'],
    'solver': ['sgd', 'adam']
}
mlp_model = MLPClassifier(random_state=42)
grid_mlp = GridSearchCV(mlp_model, param_grid_mlp, cv=3, scoring='accuracy')
grid_mlp.fit(X_train_scaled, y_train)
print(f"Best parameters for MLP: {grid_mlp.best_params_}")

In [ ]:
# Evaluation of all models including tuned models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced'),
    'Tuned Random Forest': grid_rf.best_estimator_,
    'SVM': SVC(random_state=42, class_weight='balanced', probability=True),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
    'Tuned MLP': grid_mlp.best_estimator_
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    print(f"{name}: Confusion Matrix:\n{cm}\nF1-Score: {f1}, AUC: {auc}")

In [ ]:
explainer = shap.TreeExplainer(models['Tuned Random Forest'], X_train_scaled, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_train_scaled, check_additivity=False)


print("/nShape of SHAP values:", np.array(shap_values).shape)
print("Shape of training data:", X_train_scaled.shape)


if isinstance(shap_values, list) and len(shap_values) > 1:
    shap_values = shap_values[1]


if shap_values.shape[1] != X_train_scaled.shape[1]:
    raise ValueError(
        f"Mismatch in SHAP values ({shap_values.shape[1]} features) and training data ({X_train_scaled.shape[1]} features)."
    )


shap.summary_plot(
    shap_values,
    X_train_scaled,
    feature_names=X.columns,
    plot_size=(10, 10)
)

In [ ]:
# Interpretation of Logistic Regression weights
print("Logistic Regression coefficients:")
print(pd.DataFrame(models['Logistic Regression'].coef_[0], index=X.columns, columns=['Coefficients']))

In [ ]:
# Plot Decision Tree
plt.figure(figsize=(20,10))
plot_tree(models['Decision Tree'], filled=True, feature_names=X.columns, max_depth=3)
plt.show()

In [ ]:
# User input for song popularity prediction
user_data = {col: 0 for col in X_train.columns}
print("Please enter the song details:")
for col in numeric_columns:
    user_data[col] = float(input(f"Enter {col}: "))

user_df = pd.DataFrame([user_data])
user_df_scaled = scaler.transform(user_df[X_train.columns])

for name, model in models.items():
    prediction = model.predict(user_df_scaled)
    probability = model.predict_proba(user_df_scaled)[:, 1] if hasattr(model, "predict_proba") else [0]
    print(f"{name} model prediction: {'Popular' if prediction[0] == 1 else 'Not Popular'}, Probability of being popular: {probability[0]:.2f}")

In [ ]:
# Recommendation system based on user input
X_scaled = scaler.transform(X)
similarity = cosine_similarity(X_scaled, user_df_scaled)
top_indices = np.argsort(similarity[:, 0])[-6:-1]
recommended_songs = data.iloc[top_indices]

print("Top 5 recommended songs based on your input:")
print(recommended_songs[['track_name', 'artist(s)_name']])